# NanoHotpotQA Cosine Similarity Evaluation

This notebook evaluates static embedding models on the NanoHotpotQA dataset using dot product / cosine similarity.

In [2]:
import math
import json
import numpy as np
from typing import Dict, List, Set, Tuple
from tqdm.notebook import tqdm

from utils.model2vec_model import get_token_level_embeddings

## 1. Utility & Data Loading Functions

In [3]:
def encode_text(text: str, normalize: bool = True) -> np.ndarray:
    """
    Returns a single, 1D pooled embedding for the given text.
    """
    emb = get_token_level_embeddings(text, "../models/snowflake_custom")

    if not isinstance(emb, np.ndarray):
        emb = np.asarray(emb)

    if emb.ndim != 2:
        raise ValueError(f"Expected [num_tokens, dim], got shape {emb.shape}")

    emb = emb.astype(np.float32)

    if emb.shape[0] == 0:
        raise ValueError("Embedding has zero tokens.")

    # Apply mean pooling to get a single dense vector [dim]
    pooled_emb = emb.mean(axis=0)

    if normalize:
        norm = np.linalg.norm(pooled_emb)
        if norm > 0:
            pooled_emb = pooled_emb / norm

    return pooled_emb

def load_local_hotpotqa_dataset(lang: str = "ro"):
    corpus_file = f"../data/nanohotpotqa-datasets/NanoHotpotQA_{lang}_corpus.jsonl"
    queries_file = f"../data/nanohotpotqa-datasets/NanoHotpotQA_{lang}_queries.jsonl"
    qrels_file = f"../data/nanohotpotqa-datasets/NanoHotpotQA_{lang}_qrels.jsonl"

    corpus = {}
    with open(corpus_file, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("text"):
                corpus[row["_id"]] = row["text"]

    queries = {}
    with open(queries_file, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("text"):
                queries[row["_id"]] = row["text"]

    qrels: Dict[str, Set[str]] = {}
    with open(qrels_file, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            qid = row.get("query-id") or str(row.get("query_id", ""))
            cid = row.get("corpus-id") or str(row.get("corpus_id", ""))

            if not qid or not cid:
                continue

            if qid not in qrels:
                qrels[qid] = set()

            if isinstance(cid, list):
                qrels[qid].update(cid)
            else:
                qrels[qid].add(cid)

    return corpus, queries, qrels

## 2. Dot Product Scoring & Ranking

In [4]:
def dot_score(query_emb: np.ndarray, doc_emb: np.ndarray) -> float:
    """
    Standard dense embedding score: dot product between query and doc vectors.
    When normalized, this is equivalent to cosine similarity.
    """
    return float(np.dot(query_emb, doc_emb))

def rank_documents(
    query_emb: np.ndarray,
    doc_embeddings: Dict[str, np.ndarray],
    top_k: int,
) -> List[Tuple[str, float]]:
    scores = []

    for doc_id, doc_emb in doc_embeddings.items():
        score = dot_score(query_emb, doc_emb)
        scores.append((doc_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

## 3. Evaluation Metrics

In [5]:
def precision_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    if k == 0:
        return 0.0

    top = ranked_ids[:k]
    hits = sum(1 for doc_id in top if doc_id in relevant)
    return hits / k

def recall_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0

    top = ranked_ids[:k]
    hits = sum(1 for doc_id in top if doc_id in relevant)
    return hits / len(relevant)

def mrr_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def dcg_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    dcg = 0.0

    for i, doc_id in enumerate(ranked_ids[:k], start=1):
        rel = 1.0 if doc_id in relevant else 0.0
        if rel:
            dcg += rel / math.log2(i + 1)

    return dcg

def ndcg_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0

    dcg = dcg_at_k(ranked_ids, relevant, k)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))

    if idcg == 0:
        return 0.0

    return dcg / idcg

def average_precision_at_k(ranked_ids: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0

    hits = 0
    total_precision = 0.0

    for i, doc_id in enumerate(ranked_ids[:k], start=1):
        if doc_id in relevant:
            hits += 1
            total_precision += hits / i

    return total_precision / min(len(relevant), k)

## 4. Main Evaluation Pipeline

In [6]:
def evaluate(
    lang: str = "ro",
    normalize: bool = True,
    top_k: int = 100,
):
    corpus, queries, qrels = load_local_hotpotqa_dataset(lang)

    print(f"Dataset: NanoHotpotQA ({lang})")
    print(f"Corpus size: {len(corpus)}")
    print(f"Queries: {len(queries)}")
    print(f"Qrels queries: {len(qrels)}")

    print("\nEncoding corpus...")
    doc_embeddings: Dict[str, np.ndarray] = {}

    for doc_id, text in tqdm(corpus.items()):
        doc_embeddings[doc_id] = encode_text(text, normalize=normalize)

    metrics = {
        "precision@1": [],
        "precision@3": [],
        "precision@5": [],
        "precision@10": [],
        "recall@1": [],
        "recall@3": [],
        "recall@5": [],
        "recall@10": [],
        "mrr@10": [],
        "ndcg@10": [],
        "map@100": [],
    }

    print("\nEvaluating queries...")

    for query_id, query_text in tqdm(queries.items()):
        if query_id not in qrels:
            continue

        query_emb = encode_text(query_text, normalize=normalize)
        ranked = rank_documents(query_emb, doc_embeddings, top_k=top_k)
        ranked_ids = [doc_id for doc_id, _ in ranked]
        relevant = qrels[query_id]

        metrics["precision@1"].append(precision_at_k(ranked_ids, relevant, 1))
        metrics["precision@3"].append(precision_at_k(ranked_ids, relevant, 3))
        metrics["precision@5"].append(precision_at_k(ranked_ids, relevant, 5))
        metrics["precision@10"].append(precision_at_k(ranked_ids, relevant, 10))

        metrics["recall@1"].append(recall_at_k(ranked_ids, relevant, 1))
        metrics["recall@3"].append(recall_at_k(ranked_ids, relevant, 3))
        metrics["recall@5"].append(recall_at_k(ranked_ids, relevant, 5))
        metrics["recall@10"].append(recall_at_k(ranked_ids, relevant, 10))

        metrics["mrr@10"].append(mrr_at_k(ranked_ids, relevant, 10))
        metrics["ndcg@10"].append(ndcg_at_k(ranked_ids, relevant, 10))
        metrics["map@100"].append(average_precision_at_k(ranked_ids, relevant, 100))

    results = {
        name: float(np.mean(values)) if values else 0.0
        for name, values in metrics.items()
    }

    print("\nResults:")
    for name, value in results.items():
        print(f"{name}: {value:.4f}")

    print("\nPrimary metric:")
    print(f"ndcg@10: {results['ndcg@10']:.4f}")

    return results

## 5. Execution
Modify the configuration parameters below instead of using command line arguments.

In [8]:
# Configuration parameters
LANG = "ro"              # Choices: "ro", "en"
NORMALIZE = True         # Set to False to disable L2 normalization of token embeddings
TOP_K = 100              # Number of top documents to retrieve

# Run evaluation
results = evaluate(
    lang=LANG,
    normalize=NORMALIZE,
    top_k=TOP_K
)

Dataset: NanoHotpotQA (ro)
Corpus size: 5090
Queries: 50
Qrels queries: 50

Encoding corpus...


  0%|          | 0/5090 [00:00<?, ?it/s]


Evaluating queries...


  0%|          | 0/50 [00:00<?, ?it/s]


Results:
precision@1: 0.3200
precision@3: 0.2133
precision@5: 0.1440
precision@10: 0.0900
recall@1: 0.1600
recall@3: 0.3200
recall@5: 0.3600
recall@10: 0.4500
mrr@10: 0.4288
ndcg@10: 0.3672
map@100: 0.3038

Primary metric:
ndcg@10: 0.3672
